# Auswertung Zelasto Studie

![Bild Depth Touch](images/depth-touch.jpg)

Jupyter Notebook um die in der Zelasto Studie geschriebenen CSV Dateien auf und auszuwerten. 

1. [Studienablauf](#Studienablauf)
1. [Aktuelle Lösung](Aktuelle-Lösung)
1. [Struktur Logfiles](#Struktur-Logfiles)
1. [Werte im Logfile](#Werte-im-Logfile)
1. [Statusabfolge](#Statusabfolge)
1. [Definition der Tiefenebenen](#Definition-der-Tiefenebenen)

## Studienablauf
---

* 6 Probe-Aufgaben (TASK) mit jeweils 2 Wiederholungen (TRIAL)
* 3 Blöcke (BLOCK) mit jeweils
    * 18 Aufgaben (TASK) mit jeweils 5 Wiederholungen (TRIAL)
    * In dem Block wurde immer eine Mapping-Methode (MAPPING_METHOD) genutzt
        * `direct / densening / widening`
    * Variiert wurden in den Aufgaben (TASK) die Anzahl der Tiefenebenen:
        * `6 / 9 / 12 / 15 / 18 / 21`
    * Variiert in den Wiederholungen (TRIALS) wurden die Zielebenen (random)
* Zuordnung Ebene <-> MAPPING_METHOD <-> Tiefe in eigener csv (depthlayers.csv)
* Erfolgs-/Fehlermöglichkeiten:
    * `COMPLETED / FAILED / TERMINATED`

## Aktuelle Lösung
---

* Iterativer / objektorientierter Ansatz (nicht wirklich Data Science strukturiert)
* Gesucht: besser anpassbare, „einfache“ data science Lösung

## Struktur Logfiles
---

`2021-05-11T14:55:08.419Z; INTERACTION; direct;18;14,9,5,1,17;18;1; 0.5099127;0.5482645;-0.449999869;637563489084131200;1;8`

| DateTime (LogServer)     | STATE          | mapping method | Task-No. | Target Layers | Layer-Count | Trial-Idx | PosX             | PosY                                                  | PosZ         | TimeStamp (Server) | InteractionType | CurrentLayer |
| ------------------------ | -------------- | -------------- | -------- | ------------- | ----------- | --------- | ---------------- | ----------------------------------------------------- | ------------ | ------------------ | --------------- | ------------ |
| 2021-05-11T14:55:08.419Z | INTERACTION    | direct         | 18       | 14,9,5,1,17   | 18          | 1         | 0.5099127        | 0.5482645                                             | -0.449999869 | 637563489084131200 | 1               | 8            |
| 2021-05-10T12:14:37.259Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_DESCRIPTION |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.259Z | TASK           | direct         | 2        | 7,8,1,5,2     | 9           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:13:51.730Z | MAPPING_METHOD | direct         | 1        | 4,5,3,1,2     | 6           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.264Z | SUBTASK        | direct         | 2        | 7,8,1,5,2     | 9           | 0         | 2                |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.265Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | START            |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:39.587Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_VIEW        |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:45.064Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | HOLD             |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:46.565Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | COMPLETED        |                                                       |              |                    |                 |              |
| 2021-05-10T12:10:39.115Z | SUBTASK_STATE  | direct         | 1        | 1,5           | 6           | 0         | FAILED           | wrong level approved                                  |              |                    |                 |              |
| 2021-05-10T12:13:38.243Z | SUBTASK_STATE  | densening      | 6        | 3,12          | 14          | 0         | TERMINATED       | switched to other level before hold timeout completed |              |                    |                 |              |

## Werte im Logfile
---

* Tasks:
    * 1-6 (Test)
    * 1-18 (Block 1)
    * 19-36 (Block 2)
    * 37 - 54 (Block 3)

* 3 Blocks for any mapping method

* STATE:
  
    | State Value        | Description                                       | SubTypes           |
    | ------------------ | ------------------------------------------------- | ------------------ |
    | __INTERACTION__    | Trial interaction                                 | -                  |
    | __VIEW__           | switched view (describe Task)                     | TASK_VIEW          |
    |                    | switched view in test runs (describe Task)        | Test Run TASK_VIEW |
    |                    |                                                   | TASK_DESCRIPTION   |
    | __TASK__           |                                                   |                    |
    | __MAPPING_METHOD__ | starting to next large block (mapping method)     |                    |
    | __SUBTASK__        | same as the next: starting to next task           |                    |
    | __SUBTASK_STATE__  | start of the trial after Subtask                  | START              |
    |                    | dwell time in a layer exceeded: hold-timer starts | HOLD               |
    |                    | end of the trial (Success)                        | COMPLETED          |
    |                    | end of the trial (hold failure                    | TERMINATED         |
    |                    | end of the trial (wrong level ?)                  | FAILED             |

* mapping method - describes, how layers are aligned:
    * __direct__ (equivalent distance)
    * __densening__ (larger distance on top, decreasing with depth value)
    * __widening__ (narrower distance on top, increasing with depth value)

* Task-No.
    * running number of tasks

* Target Layers, Trial-Idx
    * array of targets for each layer in current Task
    * trial-Index (zero based) describes target layer for current trial

* Layer-Count
    * number of max layers in Task

* PosX, PosY, PosZ
    * Positions received from Tracking Server
    * PosX / PosY in range [0, 1]
    * PosZ in range [-1, 1] with 0 = on the surface / no interaction, -1 max push, +1 max pull

* Timestamp (Server)
    * timestamp received from Tracking server: miliseconds based unix time stamp

* InteractionType
    * type recognized from server (1 = PUSH)

* current layer
    * layer associated with received depth value

## Statusabfolge
---

1. START (missing on first trial !) OR
    * start of an new task
2. TASK_VIEW / TASK_DESCRIPTION
3. HOLD
4. COMPLETED / TERMINATED / FAILED
   

## Definition der Tiefenebenen
---

__Location:__ `data/depthlayers.csv` [File](data/depthlayers.csv)

`6; direct; 0 | 1 | 2 | 3 | 4 | 5 | 6; 0 | 0.0834 | 0.25 | 0.4167 | 0.5834 | 0.75 | 0.9167 | 1`

| NUM_LAYERS       | MAPPING_METHOD            | DEPTH_LAYER_ID                         | DEPTH_LAYER_BORDER              |
| ---------------- | ------------------------- | -------------------------------------- | ------------------------------- |
| number of layers | mapping method for  block | idx of the depth layer in border array | start depths for each layer idx |

* mapping of layers to depth ranges for better evaluation how "good" a depth layer has been hit


## Aufgaben
---

### Vorverarbeitung - Cleaning

* Probe-Aufgaben und eigentliche Studie in verschiedene Dateien
* Ersetzen: Pro Task, muss im ersten Trial  TASK_VIEW mit START getauscht werden
* Spalten benennen (für bessere Adressierung)
* Nebenbedingung/Bonus: sowohl „nur Studie“ / „nur Test“ / „Test und Studie“ zusammen laden  Ordner-Struktur / Namensschema ?
* Lösung/Code dokumentieren (jupyter notebook)


In [1]:
# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os

export = "../export/"

path = "../data/"
all_files = glob.glob(path + "/*.csv")

# for debugging: only use first file
# all_files = all_files[9:11]

study = []
cleanedStudy = []
columnNames =["DateTime", "State", "mappingMethod", "TaskNo", "TargetLayers", "LayerCount", "TrialIdx", "posX", "posY", "posZ", "TimeStamp", "iteractionType", "currentLayer"]

totalLength = 0

for filename in all_files:
    df = pd.read_csv(filename, sep=";", names = columnNames)
    fn = os.path.basename(filename).split(".")[0]    
    df["Proband"] = fn
    study.append(df)
    totalLength += len(df)
    
print("totallines: " + str(totalLength))

data_complete = pd.concat(study, ignore_index=True)
display (data_complete)

currentLine = 0
swaps = 0

marker = []
marker.append(["Id", "EndInit", "EndTests", "SwapLines"])

for testData in study:   
    currentLine += len(testData)
    
    # compare taskNo of next row by shifting series    
    testData["shifted"] = testData["TaskNo"].shift(-1)
    
    # find elements where the taskNo of the next row is 1 and the current row is larger than 1 (that is where the learning tasks end / the whole test restarts )
    markers = testData[(testData["shifted"] == 1) & (testData["TaskNo"] > 1)]
    
    # assert that there are always exactly 2 markers 
    if len(markers) != 2:
        raise Exception("invalid number of markers")
        
    start = markers.index[0]+1
    end = markers.index[1]+1
        
    filteredData = testData.drop(testData.index[end:len(testData)])     
    filteredData = filteredData.drop(filteredData.index[0:start])
    
    filteredData.reset_index(drop=True, inplace=True)
    
    # display(filteredData)
    
    marker.append([testData["Proband"][0], start, end, "-"])    
    
    print("Processed: " + str(currentLine) + " of " +  str(totalLength) + "(" + str(testData["Proband"][0]) + ")")      

    # exchange TASK_VIEW ans START in the first Trial for each Task
   
    startLine = filteredData[(filteredData["TrialIdx"] == 0) & (filteredData["posX"] == " START")]
    taskLine= filteredData[(filteredData["TrialIdx"] == 0) & (filteredData["posX"] == " TASK_VIEW")]
    
    # iterate over task lines (start/task swpa is not necessary every time) to find START and TASK_VIEW 
    # Swap lines when TASK_VIEW was found
    for i in range(0, len(taskLine)):
        linecopy = filteredData.iloc[startLine.index[i]]
        filteredData.iloc[startLine.index[i]] = filteredData.iloc[taskLine.index[i]]
        filteredData.iloc[taskLine.index[i]] = linecopy
        
        marker.append([testData["Proband"][0], "-", "-", str(startLine.index[i] + end) + " <-> " + str(taskLine.index[i] + end)]);

    cleanedStudy.append(filteredData)
    
markerframe = pd.DataFrame(data=marker)
markerframe.to_csv(rf'{export}marker.csv', sep= ";")
    
print('finished')

df_cleanedStudy = pd.concat(cleanedStudy)
df_cleanedStudy.to_csv(rf'{export}cleanedStudy.csv', sep= ";")

totallines: 2733080


,DateTime,State,mappingMethod,TaskNo,TargetLayers,LayerCount,TrialIdx,posX,posY,posZ,TimeStamp,iteractionType,currentLayer,Proband
0,2021-05-10T12:08:25.383Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,01
1,2021-05-10T12:08:25.414Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,01
2,2021-05-10T12:08:25.446Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,01
3,2021-05-10T12:08:25.475Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,01
4,2021-05-10T12:08:25.521Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2733075,2021-06-30T12:37:47.525Z,INTERACTION,densening,1,"8,2,7,1,5",9,0,-,-,-,-,-,-,24
2733076,2021-06-30T12:37:47.564Z,INTERACTION,densening,1,"8,2,7,1,5",9,0,-,-,-,-,-,-,24
2733077,2021-06-30T12:37:47.596Z,INTERACTION,densening,1,"8,2,7,1,5",9,0,-,-,-,-,-,-,24
2733078,2021-06-30T12:37:47.639Z,INTERACTION,densening,1,"8,2,7,1,5",9,0,-,-,-,-,-,-,24


Processed: 86106 of 2733080(01)
Processed: 227176 of 2733080(02)
Processed: 339055 of 2733080(03)
Processed: 478493 of 2733080(04)
Processed: 594798 of 2733080(05)
Processed: 720103 of 2733080(06)
Processed: 883086 of 2733080(07)
Processed: 1064088 of 2733080(08)
Processed: 1184333 of 2733080(09)
Processed: 1299948 of 2733080(10)
Processed: 1391165 of 2733080(11)
Processed: 1490464 of 2733080(12)
Processed: 1674368 of 2733080(13)
Processed: 1777237 of 2733080(14)
Processed: 1904586 of 2733080(15)
Processed: 2014642 of 2733080(16)
Processed: 2120240 of 2733080(17)
Processed: 2227439 of 2733080(18)
Processed: 2343047 of 2733080(19)
Processed: 2444664 of 2733080(20)
Processed: 2543798 of 2733080(22)
Processed: 2634877 of 2733080(23)
Processed: 2733080 of 2733080(24)
finished


---

### Extraktion

#### Erfolg/Fehler für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | TARGET_RELATIVE | RESULT    |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | --------------- | --------- |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 0.46666         | COMPLETED |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |

---

In [2]:
import numpy as np
#create new Dataframe for the results per Trial and Proband
# cn_result = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Result", "Result_Numeric", "FrameNumber_Start", "FrameNumber_Finish"]

probands = []

for proband in cleanedStudy:

    start = proband[(proband['posX']  == " START")]
    # remove duplicate labels (keep last)
    start = start.drop_duplicates(['TaskNo', 'TrialIdx'], keep='last')
    
    finish = proband[(proband["posX"] == " COMPLETED") | (proband["posX"] == " FAILED") | (proband["posX"] == " TERMINATED")]
    
    # remove duplicate labels (keep first)
    finish = finish.drop_duplicates(['TaskNo', 'TrialIdx'])
    
    # setting the indices - associate the date sets with each other - nth start <-> nth finish 
    finish["FrameNumber_Start"] = start.index
    start["FrameNumber_Finish"] = finish.index
    
    finish = finish.reset_index()
    start = start.reset_index()
    
    result = pd.DataFrame(start["mappingMethod"])
    result = result.rename(columns= {"mappingMethod": "MappingMethod"})
    
    result["FrameNumber_Start"] = finish["FrameNumber_Start"]
    result["FrameNumber_Finish"] = start["FrameNumber_Finish"]
    result["Block"] = (start["TaskNo"] / 18).apply(np.ceil).astype(int)
    result["Task"] = (start["TaskNo"] - ((result["Block"] - 1) * 18)).astype(int)
    result["Trial"] = start["TrialIdx"].astype(int)
    result["NumLayers"] = start["LayerCount"]    
    result["Target"] = start["TargetLayers"].str.split(",")
    result["Target"] = result.apply(lambda item: item["Target"][item["Trial"]], axis = 1).astype(int)
    result["Target_Relative"] = result["Target"].astype(float) / result["NumLayers"].astype(float)

    result["Result"] = finish["posX"]
    result["Result_Numeric_Completed"] = result["Result"].map({" COMPLETED": 1, " FAILED": 0, " TERMINATED": 0})
    result["Result_Numeric_Terminated"] = result["Result"].map({" COMPLETED": 0, " FAILED": 0, " TERMINATED": 1})
    result["Result_Numeric_Failed"] = result["Result"].map({" COMPLETED": 0, " FAILED": 1, " TERMINATED": 0})
    
    result["Proband"] = start["Proband"]
    
    probands.append(result)

allResults = pd.concat(probands, ignore_index=True)
display(allResults)

allResults.to_csv(rf'{export}results.csv', sep= ";")


,MappingMethod,FrameNumber_Start,FrameNumber_Finish,Block,Task,Trial,NumLayers,Target,Target_Relative,Result,Result_Numeric_Completed,Result_Numeric_Terminated,Result_Numeric_Failed,Proband
0,direct,135,330,1,1,0,6,4,0.666667,COMPLETED,1,0,0,01
1,direct,413,531,1,1,1,6,5,0.833333,TERMINATED,0,1,0,01
2,direct,603,804,1,1,2,6,3,0.500000,COMPLETED,1,0,0,01
3,direct,870,969,1,1,3,6,1,0.166667,TERMINATED,0,1,0,01
4,direct,1029,1177,1,1,4,6,2,0.333333,COMPLETED,1,0,0,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,widening,89346,89495,3,18,0,6,4,0.666667,COMPLETED,1,0,0,24
6206,widening,89542,89684,3,18,1,6,5,0.833333,COMPLETED,1,0,0,24
6207,widening,89739,89833,3,18,2,6,2,0.333333,TERMINATED,0,1,0,24
6208,widening,89887,90030,3,18,3,6,3,0.500000,COMPLETED,1,0,0,24


---
#### Versuchsdauer für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .     | .              | .          | .      | .        |
| .       | .     | .    | .     | .              | .          | .      | .        |

---

In [3]:
#create new Dataframe for the results per Trial and Proband
# cn_times = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Duration"]
# end_times = pd.DataFrame(columns = cn_times) 

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

probands = []

# iterate through all probands
for proband in cleanedStudy:

    start = proband[(proband['posX']  == " START")]
    # remove duplicate labels (keep last)
    start = start.drop_duplicates(['TaskNo', 'TrialIdx'], keep='last')
    
    finish = proband[(proband["posX"] == " COMPLETED") | (proband["posX"] == " FAILED") | (proband["posX"] == " TERMINATED")]
    
    # remove duplicate labels (keep first)
    finish = finish.drop_duplicates(['TaskNo', 'TrialIdx'])
    
    # setting the indices - associate the date sets with each other - nth start <-> nth finish 
    finish["FrameNumber_Start"] = start.index
    start["FrameNumber_Finish"] = finish.index
    
    start["DateTime"] = proband["DateTime"].astype("string")
    finish["DateTime"] = proband["DateTime"].astype("string")
    
    finish = finish.reset_index()
    start = start.reset_index()
    
    result = pd.DataFrame(start["mappingMethod"])
    result = result.rename(columns= {"mappingMethod": "MappingMethod"})
    
    result["FrameNumber_Start"] = finish["FrameNumber_Start"]
    result["FrameNumber_Finish"] = start["FrameNumber_Finish"]
    result["Block"] = (start["TaskNo"] / 18).apply(np.ceil).astype(int)
    result["Task"] = (start["TaskNo"] - ((result["Block"] - 1) * 18)).astype(int)
    result["Trial"] = start["TrialIdx"].astype(int)
    result["NumLayers"] = start["LayerCount"]    
    result["Target"] = start["TargetLayers"].str.split(",")
    result["Target"] = result.apply(lambda item: item["Target"][item["Trial"]], axis = 1).astype(int)
    result["Target_Relative"] = result["Target"].astype(float) / result["NumLayers"].astype(float)

    result["Duration"] = (pd.to_datetime(finish["DateTime"], format=timeFormat) - pd.to_datetime(start["DateTime"], format=timeFormat)).dt.total_seconds()
    result["Result"] = finish["posX"]
    
    result["Proband"] = start["Proband"]
    
    probands.append(result)

allTimes = pd.concat(probands, ignore_index=True)
display(allTimes)

allTimes.to_csv(rf'{export}times.csv', sep= ";")


,MappingMethod,FrameNumber_Start,FrameNumber_Finish,Block,Task,Trial,NumLayers,Target,Target_Relative,Duration,Result,Proband
0,direct,135,330,1,1,0,6,4,0.666667,12.221,COMPLETED,01
1,direct,413,531,1,1,1,6,5,0.833333,4.414,TERMINATED,01
2,direct,603,804,1,1,2,6,3,0.500000,7.155,COMPLETED,01
3,direct,870,969,1,1,3,6,1,0.166667,3.627,TERMINATED,01
4,direct,1029,1177,1,1,4,6,2,0.333333,5.516,COMPLETED,01
...,...,...,...,...,...,...,...,...,...,...,...,...
6205,widening,89346,89495,3,18,0,6,4,0.666667,7.359,COMPLETED,24
6206,widening,89542,89684,3,18,1,6,5,0.833333,5.325,COMPLETED,24
6207,widening,89739,89833,3,18,2,6,2,0.333333,3.438,TERMINATED,24
6208,widening,89887,90030,3,18,3,6,3,0.500000,5.467,COMPLETED,24


#### Versuchsdauer für jeden Trial und je Proband (Trial als eigene Spalten)

| PROBAND | BLOCK | TASK | TRIAL0 | ... | TRIALN | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | ------ | --- | -------| -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | 5.532s | ... | 2.546s | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .      | ... | .      | .              | .          | .      | .        |
| .       | .     | .    | .      | ... | .      | .              | .          | .      | .        |

In [4]:
pivot = allTimes.pivot(index=["Proband", "Block", "Task", "MappingMethod", "NumLayers"], columns="Trial", values="Duration")

# print(end_times.columns)
# print(end_times.index)

# display(pivot.columns.names)
# print(pivot.index.names)

# creating the multiIndex... 
m_idx = pd.MultiIndex.from_frame(pivot)

# print(m_idx.get_level_values(0))
    
pivot["Duration_Sum"] = m_idx.get_level_values(0) +  m_idx.get_level_values(1) +  m_idx.get_level_values(2) +  m_idx.get_level_values(3) +  m_idx.get_level_values(4)

display(pivot)

pivot.to_csv(rf'{export}times_repeat.csv',sep= ";")

Trial                                            0       1       2       3  \
Proband Block Task MappingMethod NumLayers                                   
01      1     1     direct       6          12.221   4.414   7.155   3.627   
              2     direct       9           9.300   6.029  13.550  11.628   
              3     direct       9          11.357   7.849   6.402   4.394   
              4     direct       15         10.526   7.910   7.202   7.145   
              5     direct       12          9.006   8.612   8.256   5.049   
...                                            ...     ...     ...     ...   
24      3     14    widening     9           7.163   4.929   5.777   6.530   
              15    widening     12         10.430   5.419   8.240   5.712   
              16    widening     12         10.177  15.856   6.571   6.755   
              17    widening     21         12.128   9.413   4.488   4.400   
              18    widening     6           7.359   5.325   3.438   5.467   

Trial                                           4  Duration_Sum  
Proband Block Task MappingMethod NumLayers                       
01      1     1     direct       6          5.516        32.933  
              2     direct       9          9.832        50.339  
              3     direct       9          8.062        38.064  
              4     direct       15         7.956        40.739  
              5     direct       12         4.624        35.547  
...                                           ...           ...  
24      3     14    widening     9          5.683        30.082  
              15    widening     12         6.656        36.457  
              16    widening     12         5.323        44.682  
              17    widening     21         6.644        37.073  
              18    widening     6          5.088        26.677  

[1242 rows x 6 columns]

#### Ergebnis für jeden Trial und je Proband (Trial als eigene Spalten)

| PROBAND | BLOCK | TASK | TRIAL0    | ... | TRIALN | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | --------- | --- | -------| -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | COMPLETED | ... | FAILED | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .         | ... | .      | .              | .          | .      | .        |
| .       | .     | .    | .         | ... | .      | .              | .          | .      | .        |

In [5]:
pivot = allResults.pivot(index=["Proband", "Block", "Task", "MappingMethod", "NumLayers"], columns="Trial", values=["Result", "Result_Numeric_Completed", "Result_Numeric_Failed", "Result_Numeric_Terminated"] )

# print(end_times.columns)
# print(end_times.index)

# display(pivot.columns.names)
# print(pivot.index.names)

# creating the multiIndex... 
m_idx = pd.MultiIndex.from_frame(pivot)

# print(m_idx.get_level_values(0))

pivot['rate_completed'] = (m_idx.get_level_values(5) +  m_idx.get_level_values(6) +  m_idx.get_level_values(7) +  m_idx.get_level_values(8) +  m_idx.get_level_values(9)) / 5
pivot['rate_failed'] = (m_idx.get_level_values(10) +  m_idx.get_level_values(11) +  m_idx.get_level_values(12) +  m_idx.get_level_values(13) +  m_idx.get_level_values(14)) / 5
pivot['rate_terminated'] = (m_idx.get_level_values(15) +  m_idx.get_level_values(16) +  m_idx.get_level_values(17) +  m_idx.get_level_values(18) +  m_idx.get_level_values(19)) / 5

pivot['rate_sum'] = pivot['rate_completed'] + pivot['rate_failed'] + pivot['rate_terminated']

display(pivot)

pivot.to_csv(rf'{export}results_repeat.csv',sep= ";")

Result               \
Trial                                                0            1   
Proband Block Task MappingMethod NumLayers                            
01      1     1     direct       6           COMPLETED   TERMINATED   
              2     direct       9           COMPLETED    COMPLETED   
              3     direct       9           COMPLETED    COMPLETED   
              4     direct       15          COMPLETED    COMPLETED   
              5     direct       12          COMPLETED    COMPLETED   
...                                                ...          ...   
24      3     14    widening     9              FAILED   TERMINATED   
              15    widening     12          COMPLETED   TERMINATED   
              16    widening     12          COMPLETED    COMPLETED   
              17    widening     21          COMPLETED    COMPLETED   
              18    widening     6           COMPLETED    COMPLETED   

                                                                      \
Trial                                                 2            3   
Proband Block Task MappingMethod NumLayers                             
01      1     1     direct       6            COMPLETED   TERMINATED   
              2     direct       9           TERMINATED    COMPLETED   
              3     direct       9            COMPLETED   TERMINATED   
              4     direct       15          TERMINATED   TERMINATED   
              5     direct       12           COMPLETED   TERMINATED   
...                                                 ...          ...   
24      3     14    widening     9            COMPLETED    COMPLETED   
              15    widening     12           COMPLETED       FAILED   
              16    widening     12           COMPLETED    COMPLETED   
              17    widening     21          TERMINATED   TERMINATED   
              18    widening     6           TERMINATED    COMPLETED   

                                                         \
Trial                                                 4   
Proband Block Task MappingMethod NumLayers                
01      1     1     direct       6            COMPLETED   
              2     direct       9           TERMINATED   
              3     direct       9            COMPLETED   
              4     direct       15          TERMINATED   
              5     direct       12          TERMINATED   
...                                                 ...   
24      3     14    widening     9            COMPLETED   
              15    widening     12           COMPLETED   
              16    widening     12          TERMINATED   
              17    widening     21              FAILED   
              18    widening     6               FAILED   

                                           Result_Numeric_Completed           \
Trial                                                             0  1  2  3   
Proband Block Task MappingMethod NumLayers                                     
01      1     1     direct       6                                1  0  1  0   
              2     direct       9                                1  1  0  1   
              3     direct       9                                1  1  1  0   
              4     direct       15                               1  1  0  0   
              5     direct       12                               1  1  1  0   
...                                                             ... .. .. ..   
24      3     14    widening     9                                0  0  1  1   
              15    widening     12                               1  0  1  0   
              16    widening     12                               1  1  1  1   
              17    widening     21                               1  1  0  0   
              18    widening     6                                1  1  0  1   

                                               ... Result_Numeric_Failed  \
Trial               

---
#### Erfolg/Fehlerquote über alle Probanden je Ebenenanzahl

| NUM_LAYERS     | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---       | ---     | ---        | --- |
| 5              | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .              | .         | .       | .          | .   |
| .              | .         | .       | .          | .   |
---

In [35]:
df_resultStat = pd.read_csv(rf'{export}results.csv', sep=";")
# df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["NumLayers", "Result"]) 

groups = df_resultStat.groupby(["NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0], 0, 0, 0, 0, 0, 0, 0]

count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["NumLayers"] == index[0])]
    r_index = rate.index
        
    if index[1] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 1] = compl_sum
        
    if index[1] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 3] = fail_sum
        
    if index[1] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 5] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by="NumLayers", key=lambda col: col.astype(int)))

df_rates.to_csv(rf'{export}rates_global_layers.csv', sep= ";")

,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,6,741,0.715942,13,0.0125604,281,0.271498,1035,1
1,9,771,0.744928,16,0.0154589,248,0.239614,1035,1
2,12,732,0.707246,15,0.0144928,288,0.278261,1035,1
3,15,721,0.696618,27,0.026087,287,0.277295,1035,1
4,18,708,0.684058,27,0.026087,300,0.289855,1035,1
5,21,676,0.65314,37,0.0357488,322,0.311111,1035,1


#### Dauer über alle Probanden je Ebenenanzahl

| NUM_LAYERS     | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---       | ---     | ---        | --- |
| 5              | 2 / 40s   | 5 / 80s | 3 / 60s    | 10  |
| .              | .         | .       | .          | .   |
| .              | .         | .       | .          | .   |
---

In [49]:
df_resultStat = pd.read_csv(rf'{export}times.csv', sep=";")
# df_resultStat = df_resultStat.iloc[1: , :]

cn_times = ["NumLayers", "Sum_Num", "Sum_Total", "All_Avg"]
df_times = pd.DataFrame() 

# select all combinations of mappng method / num layers

# group all byNumLayers and Result
# compute sum and count for each combination
# pivot results into columns
grouped = df_resultStat.groupby(["NumLayers", "Result"])["Duration"].agg(["sum", "count", "mean"]).reset_index().pivot(index="NumLayers", columns="Result")
m_idx = pd.MultiIndex.from_frame(grouped)

# display(grouped)

# now only for formatting and additional sums: extract multi index values 
df_times["NumLayers"] = grouped.index

df_times["Completed_Num"] = m_idx.get_level_values(3)
df_times["Completed_Total"] = m_idx.get_level_values(0)
df_times["Completed_Avg"] = m_idx.get_level_values(6)

df_times["Failed_Num"] = m_idx.get_level_values(4)
df_times["Failed_Total"] = m_idx.get_level_values(1)
df_times["Failed_Avg"] = m_idx.get_level_values(7)

df_times["Terminated_Num"] = m_idx.get_level_values(5)
df_times["Terminated_Total"] = m_idx.get_level_values(2)
df_times["Terminated_Avg"] = m_idx.get_level_values(8)

df_times["Sum_Num"] = df_times["Completed_Num"] + df_times["Failed_Num"] + df_times["Terminated_Num"]
df_times["Sum_Total"] = df_times["Completed_Total"] + df_times["Failed_Total"] + df_times["Terminated_Total"]
df_times["All_Avg"] = df_times["Sum_Total"] / df_times["Sum_Num"]


display(df_times)

df_times.to_csv(rf'{export}times_global_layers.csv', sep= ";")


,NumLayers,Completed_Num,Completed_Total,Completed_Avg,Failed_Num,Failed_Total,Failed_Avg,Terminated_Num,Terminated_Total,Terminated_Avg,Sum_Num,Sum_Total,All_Avg
0,6,741,7459.320,10.066559,13,156.503,12.038692,281,1482.331,5.275199,1035,9098.154,8.790487
1,9,771,9619.059,12.476082,16,134.370,8.398125,248,1967.014,7.931508,1035,11720.443,11.324100
2,12,732,8455.875,11.551742,15,110.883,7.392200,288,2048.191,7.111774,1035,10614.949,10.255989
3,15,721,7908.313,10.968534,27,374.777,13.880630,287,3508.049,12.223167,1035,11791.139,11.392405
4,18,708,9478.806,13.388144,27,431.385,15.977222,300,3524.329,11.747763,1035,13434.520,12.980213
5,21,676,9325.736,13.795467,37,455.933,12.322514,322,2919.135,9.065637,1035,12700.804,12.271308


---
#### Erfolg/Fehlerquote über je Probanden je Ebenenanzahl

| Proband     | NUM_LAYERS     | COMPLETED | FAILED  | TERMINATED | SUM |
| ---         | ---            | ---       | ---     | ---        | --- |
| 0           | 5              | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .           | .              | .         | .       | .          | .   |
| .           | .              | .         | .       | .          | .   |
---

In [50]:
df_resultStat = pd.read_csv(rf'{export}results.csv', sep=";")
# df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["Proband", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["Proband", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], 0, 0, 0, 0, 0, 0, 0]

    
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["Proband"] == index[0]) & (df_rates["NumLayers"] == index[1])]
    r_index = rate.index
        
    if index[2] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 2] = compl_sum
        
    if index[2] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 4] = fail_sum
        
    if index[2] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 6] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["Proband", "NumLayers"], key=lambda col: col.astype(int)))

df_rates.to_csv(rf'{export}rates_subject_layers.csv', sep= ";")

,Proband,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,1,6,27,0.6,0,0,18,0.4,45,1
1,1,9,31,0.688889,0,0,14,0.311111,45,1
2,1,12,29,0.644444,1,0.0222222,15,0.333333,45,1
3,1,15,33,0.733333,2,0.0444444,10,0.222222,45,1
4,1,18,31,0.688889,1,0.0222222,13,0.288889,45,1
...,...,...,...,...,...,...,...,...,...,...
133,24,9,31,0.688889,4,0.0888889,10,0.222222,45,1
134,24,12,30,0.666667,2,0.0444444,13,0.288889,45,1
135,24,15,27,0.6,4,0.0888889,14,0.311111,45,1
136,24,18,27,0.6,7,0.155556,11,0.244444,45,1


---
#### Erfolg/Fehlerquote über alle Probanden je Ebenenanzahl und mapping methode

| MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---        | ---       | ---     | ---        | --- |
| direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .              | .          | .         | .       | .          | .   |
| .              | .          | .         | .       | .          | .   |
---

In [8]:
df_resultStat = pd.read_csv(rf'{export}results.csv', sep=";")
#df_resultStat = df_resultStat.iloc[0: , :]

cn_rates = ["MappingMethod", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["MappingMethod", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], 0, 0, 0, 0, 0, 0, 0]
    
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["MappingMethod"] == index[0]) & (df_rates["NumLayers"] == index[1])]
    r_index = rate.index
        
    if index[2] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 2] = compl_sum
        
    elif index[2] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 4] = fail_sum
        
    elif index[2] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 6] = term_sum
        
    else:
        print(index[2])
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["MappingMethod", "NumLayers"]))

df_rates.to_csv(rf'{export}rates_global_mapping_layers.csv', sep= ";")

,MappingMethod,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,densening,6,254,0.736232,3,0.00869565,88,0.255072,345,1
1,densening,9,276,0.8,6,0.0173913,63,0.182609,345,1
2,densening,12,261,0.756522,1,0.00289855,83,0.24058,345,1
3,densening,15,262,0.75942,3,0.00869565,80,0.231884,345,1
4,densening,18,263,0.762319,4,0.0115942,78,0.226087,345,1
5,densening,21,239,0.692754,5,0.0144928,101,0.292754,345,1
6,direct,6,255,0.73913,2,0.0057971,88,0.255072,345,1
7,direct,9,275,0.797101,3,0.00869565,67,0.194203,345,1
8,direct,12,255,0.73913,4,0.0115942,86,0.249275,345,1
9,direct,15,254,0.736232,8,0.0231884,83,0.24058,345,1


---
#### Erfolg/Fehlerquote über je Proband je Ebenenanzahl und mapping methode

| Proband | MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---     | ---            | ---        | ---       | ---     | ---        | --- |
| 0       | direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .       | .              | .          | .         | .       | .          | .   |
| .       | .              | .          | .         | .       | .          | .   |
---

In [9]:
df_resultStat = pd.read_csv(rf'{export}results.csv', sep=";")
# df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["Proband", "MappingMethod", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], elem[0][2], 0, 0, 0, 0, 0, 0, 0]

count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["Proband"] == index[0]) & (df_rates["MappingMethod"] == index[1]) & (df_rates["NumLayers"] == index[2])]
    r_index = rate.index
        
    if index[3] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 3] = compl_sum
        
    if index[3] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 5] = fail_sum
        
    if index[3] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 7] = term_sum
        
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["Proband", "MappingMethod", "NumLayers"]))

df_rates.to_csv(rf'{export}rates_subject_mapping_layers.csv', sep= ";")

,Proband,MappingMethod,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,1,densening,6,9,0.6,0,0,6,0.4,15,1
1,1,densening,9,11,0.733333,0,0,4,0.266667,15,1
2,1,densening,12,10,0.666667,0,0,5,0.333333,15,1
3,1,densening,15,14,0.933333,0,0,1,0.0666667,15,1
4,1,densening,18,10,0.666667,0,0,5,0.333333,15,1
...,...,...,...,...,...,...,...,...,...,...,...
409,24,widening,9,8,0.533333,2,0.133333,5,0.333333,15,1
410,24,widening,12,10,0.666667,1,0.0666667,4,0.266667,15,1
411,24,widening,15,8,0.533333,0,0,7,0.466667,15,1
412,24,widening,18,7,0.466667,4,0.266667,4,0.266667,15,1


---
#### Versuchsdauer über alle Probanden (nach Outcome)
| MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED | TERMINATED | AVG     |
| ---            | ---        | ---       | ---    | ---        | ---     |
| direct         | 5          | 10.145s   | 9.759s | 11.763s    | 10.345s |
| .              | .          | .         | .       | .          | .      |
| .              | .          | .         | .       | .          | .      |
---

---
#### Versuchsdauer je Proband je Ebenenanzahl und mapping methode

| Proband | MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---     | ---            | ---        | ---       | ---     | ---        | --- |
| 0       | direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .       | .              | .          | .         | .       | .          | .   |
| .       | .              | .          | .         | .       | .          | .   |
---

In [62]:
df_resultStat = pd.read_csv(rf'{export}times.csv', sep=";")
# df_resultStat = df_resultStat.iloc[1: , :]

cn_times = ["Proband", "MappingMethod", "NumLayers", "Completed_Sum", "Completed_Avg", "Completed_Num", "Failed_Sum", "Failed_Avg", "Failed_Num", "Terminated_Sum", "Terminated_Avg", "Terminated_Num", "All_Sum", "All_Avg", "All_Num"]
df_times = pd.DataFrame(columns = cn_times) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers"])

# stats = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Duration"])["Duration"].mean()

# print(stats)

# convert solumns to floats to prevent division by zero error
df_times["Completed_Sum"] = df_times["Completed_Sum"].astype('float64')
df_times["Completed_Num"] = df_times["Completed_Num"].astype('float64')
df_times["Failed_Sum"] = df_times["Failed_Sum"].astype('float64')
df_times["Failed_Num"] = df_times["Failed_Num"].astype('float64')
df_times["Terminated_Sum"] = df_times["Terminated_Sum"].astype('float64')
df_times["Terminated_Num"] = df_times["Terminated_Num"].astype('float64')
df_times["All_Sum"] = df_times["All_Sum"].astype('float64')
df_times["All_Num"] = df_times["All_Num"].astype('float64')

for elem in groups:
    df_times.loc[len(df_times)] = [elem[0][0], elem[0][1], elem[0][2], 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

count = grouped["Duration"].sum()

for index, value in count.items():
    display(value)
    
    time = df_times.loc[(df_times["Proband"] == index[0]) & (df_times["MappingMethod"] == index[1]) & (df_times["NumLayers"] == index[2])]
    t_index = time.index
        
    if index[3] == " COMPLETED":
        compl_sum = time["Completed_Sum"] + value;
        df_times.iat[t_index[0], 3] = compl_sum
        df_times.iat[t_index[0], 5] += 1
        
    if index[3] == " FAILED":
        fail_sum = time["Failed_Sum"] + value;
        df_times.iat[t_index[0], 6] = fail_sum
        df_times.iat[t_index[0], 8] += 1
        
    if index[3] == " TERMINATED":
        term_sum = time["Terminated_Sum"] + value;
        df_times.iat[t_index[0], 9] = term_sum
        df_times.iat[t_index[0], 11] += 1
        
df_times["All_Sum"] = df_times["Completed_Sum"] + df_times["Failed_Sum"] + df_times["Terminated_Sum"]
df_times["All_Num"] = df_times["Completed_Num"] + df_times["Failed_Num"] + df_times["Terminated_Num"]
df_times["Completed_Avg"] = (df_times["Completed_Sum"] / df_times["Completed_Num"]).replace(np.inf, 0).replace(np.nan, 0)
df_times["Failed_Avg"] = (df_times["Failed_Sum"] / df_times["Failed_Num"]).replace(np.inf, 0).replace(np.nan, 0)
df_times["Terminated_Avg"] = (df_times["Terminated_Sum"] / df_times["Terminated_Num"]).replace(np.inf, 0).replace(np.nan, 0)

df_times["All_Avg"] = (df_times["All_Sum"] / df_times["All_Num"]).replace(np.inf, 0).replace(np.nan, 0)
    
display(df_times.sort_values(by=["Proband", "MappingMethod", "NumLayers"]))

df_times.to_csv(rf'{export}times_subject_mapping_layers.csv', sep= ";")

65.216

25.868

75.472

16.724

298.49800000000005

...

26.329

89.916

21.195

26.825000000000003

,Proband,MappingMethod,NumLayers,Completed_Sum,Completed_Avg,Completed_Num,Failed_Sum,Failed_Avg,Failed_Num,Terminated_Sum,Terminated_Avg,Terminated_Num,All_Sum,All_Avg,All_Num
0,1,densening,6,65.216,65.216,1.0,0.000,0.000,0.0,25.868,25.868,1.0,91.084,45.542000,2.0
1,1,densening,9,75.472,75.472,1.0,0.000,0.000,0.0,16.724,16.724,1.0,92.196,46.098000,2.0
2,1,densening,12,298.498,298.498,1.0,0.000,0.000,0.0,31.782,31.782,1.0,330.280,165.140000,2.0
3,1,densening,15,112.090,112.090,1.0,0.000,0.000,0.0,6.838,6.838,1.0,118.928,59.464000,2.0
4,1,densening,18,88.586,88.586,1.0,0.000,0.000,0.0,34.726,34.726,1.0,123.312,61.656000,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,24,widening,9,49.391,49.391,1.0,15.281,15.281,1.0,28.607,28.607,1.0,93.279,31.093000,3.0
410,24,widening,12,92.473,92.473,1.0,5.712,5.712,1.0,19.163,19.163,1.0,117.348,39.116000,3.0
411,24,widening,15,57.681,57.681,1.0,0.000,0.000,0.0,47.609,47.609,1.0,105.290,52.645000,2.0
412,24,widening,18,65.097,65.097,1.0,23.920,23.920,1.0,26.329,26.329,1.0,115.346,38.448667,3.0


* Plot: über alle Probanden für eine bestimmte Konstellation aus mapping method, Ebenenanzahl, Zielebene 
    * Y: Ebenen / Tiefe
    * X: Zeit
* Bonus: Y-Achse nach realen Tiefenwerten  skaliert (depthlayers.csv)

| PROBAND | TIMESTAMP | CURRENT_LAYER | Z     |
| ---     | ---       | ---           | ---   |
| 154     | 12345678  | 6             | -0.54 |
| .       | .         | .             | .     |
| .       | .         | .             | .     |

### Bonus
---
* depthlayers.csv: Mapping der Tiefenwerte auf die zugehörige Ebene nach Mapping-Methode Achtung: Z-Werte in Studie sind im Bereich [–1,0] zu bewerten, depthlayers.csv gibt die absolut-Werte an!
* Für erfolgreiche Trials: Jeweils zwischen HOLD und COMPLETED 
    * Maximale Schwankung (% der Ebenenbreite)
    * Durchschnittstiefe (prozentualer Abstand zur Ebenenmitte) 
    * Wie „weit“ waren die Probanden von einem Fehlschlag entfernt / wie sicher haben Sie die ebene gehalten ?
* „Wackler“ vor HOLD (wie oft wurde die Zielebene erreicht, aber nicht gehalten ?)
* Dies jeweils nach Ebenenanzahl und Mapping_Method gruppiert (über alle Probanden)
* Statistische Auswertung – Vorschläge?
* Hypothesen:
    * Sweet Spot für Anzahl der Ebenen (bspw: ab wann nimmt die Fehlerquote plötzlich stark zu ? Oder: bei welcher Ebenenanzahl wird eine Genauigkeit > 95% im Schnitt erzielt ?)
    * Vergleich der Mapping_Method: 
        * H1: widening ist genauer als direct und densening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  Genauigkeit sinkt, deswegen sind die „unteren“ (aka höheren) Ebenen weiter auseinander
        * H2: densening ist genauer als direct und widening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  konstante Kraftnutzung für den Ebenenwechsel = „untere“ (aka höhere) Ebenen näher beieinander
        * H3: direct ist genauer als widening und densening - Kraft spielt keine (oder nur geringfügige) Rolle, wichtig ist die äquidistante Positionierung der Tiefenebenen


In [ ]:
# your code here



In [ ]:
print(test)